# 🔎 Aula 08 — Embeddings, Similaridade e RAG
## Guilda de IA — Introdução à IA Generativa

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/aula08_embeddings_rag_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Objetivos:**
1. Embeddings: texto como vetores
2. FAISS: vector store em memória
3. Busca semântica: similaridade por cosseno
4. Agente RAG: LLM com ferramenta de busca

**Modelos:**
- `gemma4:e2b-it-qat` — LLM (Gemma 4 E2B com QAT, suporta tool calling)
- `nomic-embed-text-v2-moe` — modelo de embedding (Matryoshka, 256 dim)

## 🏗️ Setup

⚠️ Vá em `Runtime → Change runtime type` → **T4 GPU**

Execute a célula abaixo e aguarde — leva ~2 min na primeira vez.
Vamos carregar dois modelos que coexistem na GPU T4:
- **gemma4:e2b-it-qat** (~2B params, QAT) — o LLM
- **nomic-embed-text-v2-moe** (~0.5B params) — o modelo de embedding

In [ ]:
# ── Setup: Ollama + gemma4:e2b-it-qat + nomic-embed-text-v2-moe + deps ────
# Tudo em uma célula. Execute e prossiga.

# 1. Instalar Ollama + deps Python
!apt-get install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q langchain langchain-openai langchain-core langgraph faiss-cpu sentence-transformers nest_asyncio requests

# 2. Workaround GPU Colab + keep alive
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

# 3. Iniciar servidor
!pkill -f ollama 2> /dev/null; sleep 1
import subprocess
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env={**os.environ})

# 4. Aguardar servidor
import time, requests
for i in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            break
    except:
        time.sleep(1)

# 5. Baixar modelos
!ollama pull gemma4:e2b-it-qat
!ollama pull nomic-embed-text-v2-moe

# 6. Warm up LLM
print("🔥 Warm up LLM...")
start = time.time()
!curl -s http://localhost:11434/api/chat -d '{"model":"gemma4:e2b-it-qat","messages":[{"role":"user","content":"Hi"}],"stream":false,"keep_alive":-1}' > /dev/null
print(f"✅ LLM pronto em {time.time()-start:.1f}s")

# 7. Warm up embedding model
print("🔥 Warm up embedding model...")
start = time.time()
!curl -s http://localhost:11434/api/embeddings -d '{"model":"nomic-embed-text-v2-moe","prompt":"warm up","keep_alive":-1}' > /dev/null
print(f"✅ Embedding model pronto em {time.time()-start:.1f}s")
print("\n🎉 Tudo pronto! GPU T4 com 2 modelos carregados.")

## 1. Banco de fatos

Vamos criar um pequeno banco de fatos sobre **filosofia e computação**.
Esses fatos são curtos (cabem em 512 tokens) e alguns se complementam semanticamente
— perfeito para testar busca semântica e combinação de informações.

In [ ]:
# ── Banco de fatos: filosofia + computação ──────────────────────────
# Fatos curtos, autocontidos, alguns semanticamente próximos.
# Ideal para demo de busca semântica e agente RAG.

fatos = [
    # ── Computação ──
    "Alan Turing propôs o 'Teste de Turing' em 1950, no artigo 'Computing Machinery and Intelligence', como critério para determinar se uma máquina pode pensar.",
    "A máquina de Turing é um modelo matemático abstrato de computação proposto por Alan Turing em 1936, capaz de simular qualquer algoritmo.",
    "John von Neumann descreveu a arquitetura de computador moderna em 1945, onde CPU, memória e I/O compartilham um barramento único.",
    "Edsger Dijkstra argumentou que 'a computação é a única atividade humana em que a palavra bug foi usada antes de ser inventada', destacando a natureza precisa da programação.",
    "Donald Knuth iniciou 'The Art of Computer Programming' em 1962, uma obra de referência sobre algoritmos que permanece inacabada até hoje.",
    "Tim Berners-Lee inventou a World Wide Web em 1989 no CERN, combinando hipertexto com a internet para criar um sistema de documentos interligados.",
    "Claude Shannon fundou a teoria da informação em 1948 com o artigo 'A Mathematical Theory of Communication', definindo o bit como unidade de informação.",
    "O problema P vs NP, formulado em 1971, pergunta se todo problema cuja solução pode ser verificada rapidamente também pode ser resolvida rapidamente — e permanece em aberto.",

    # ── Filosofia da mente / IA ──
    "John Searle argumentou contra a inteligência artificial forte com o experimento mental da 'Sala Chinesa' em 1980: entender símbolos não é entender significado.",
    "René Descartes formulou 'Cogito, ergo sum' (penso, logo existo) em 1637, argumentando que a dúvida prova a existência do sujeito pensante.",
    "O 'problema difícil da consciência', cunhado por David Chalmers em 1995, pergunta por que existe experiência subjetiva — o 'como se sente' — e não apenas processamento.",
    "Aristóteles definiu o silogismo como forma de raciocínio lógico em que, dadas duas premissas, uma conclusão se segue necessariamente.",
    "Platão argumentou que o conhecimento é reminiscência: aprender é recordar o que a alma já conhecia, como descrito no diálogo 'Mênon'.",

    # ── Conexões filosofia + computação ──
    "Gottfried Wilhelm Leibniz imaginou uma 'linguagem universal de pensamento' (characteristica universalis) que permitiria resolver disputas por cálculo — um sonho precursor da computação.",
    "George Boole publicou 'An Investigation of the Laws of Thought' em 1854, mostrando que a lógica pode ser tratada como álgebra — base da computação digital.",
    "Ada Lovelace escreveu em 1843 que a máquina analítica de Babbage poderia compor 'musica complexa', sendo a primeira a imaginar computação para além de números.",
    "Alan Turing e Alonzo Church independentemente provaram em 1936 que não existe algoritmo para decidir a verdade de toda afirmação matemática — a tese de Church-Turing.",
    "Ludwig Wittgenstein argumentou em 'Investigações Filosóficas' (1953) que o significado das palavras está no seu uso, não em referências fixas — influenciando a semântica distribucional moderna.",
    "Norbert Wiener cunhou 'cibernética' em 1948 como o estudo do controle e comunicação em sistemas animados e máquinas — fundando a interdisciplina entre biologia e computação.",
]

print(f"📚 {len(fatos)} fatos carregados")
print("=" * 60)
for i, fato in enumerate(fatos):
    print(f"  [{i:02d}] {fato[:80]}...")

## 2. Embeddings com nomic-embed-text-v2-moe

Vamos converter cada fato em um vetor de **256 dimensões** usando o `nomic-embed-text-v2-moe`.

Este modelo suporta **Matryoshka embeddings** — podemos truncar para 768, 512, 256, 128 dims.
Vamos usar 256: leve e suficiente para nossa demo.

In [ ]:
# ── Gerar embeddings via Ollama API ───────────────────────────────
import requests
import numpy as np

EMBED_DIM = 256  # Matryoshka: truncar em 256 dims
OLLAMA_URL = "http://localhost:11434/api/embeddings"

def gerar_embedding(texto: str, dim: int = EMBED_DIM) -> np.ndarray:
    """Gera embedding via Ollama e trunca para `dim` dimensões (Matryoshka)."""
    resp = requests.post(OLLAMA_URL, json={
        "model": "nomic-embed-text-v2-moe",
        "prompt": texto,
        "keep_alive": -1,
    })
    emb = resp.json()["embedding"]
    emb = np.array(emb[:dim], dtype=np.float32)  # truncar (Matryoshka!)
    emb = emb / np.linalg.norm(emb)  # normalizar (L2) para similaridade por cosseno
    return emb

print(f"🔄 Gerando embeddings ({EMBED_DIM} dims) para {len(fatos)} fatos...")
vetores = np.array([gerar_embedding(f) for f in fatos])
print(f"✅ Shape: {vetores.shape} — {vetores.nbytes / 1024:.1f} KB em memória")
print(f"\nExemplo — fato 0:")
print(f"  Texto: {fatos[0][:60]}...")
print(f"  Vetor: [{vetores[0][0]:.4f}, {vetores[0][1]:.4f}, {vetores[0][2]:.4f}, ...]")

## 3. Vector Store com FAISS

FAISS (Facebook AI Similarity Search) é uma biblioteca da Meta para busca de
similaridade em vetores densos. É rápida, simples, e funciona em memória.

Como normalizamos os vetores (L2), a busca por distância L2 é equivalente
a similaridade por cosseno.

In [ ]:
# ── Criar índice FAISS ─────────────────────────────────────────────
import faiss

index = faiss.IndexFlatL2(EMBED_DIM)  # índice plano, distância L2
index.add(vetores)  # adicionar todos os vetores

print(f"✅ Índice FAISS criado: {index.ntotal} vetores de {EMBED_DIM} dimensões")
print(f"   Tipo: IndexFlatL2 (busca exata, força bruta)")
print(f"   Em memória — ideal para demos e datasets pequenos")

## 4. Busca semântica

Agora vamos buscar! Fazemos uma pergunta, geramos o embedding da pergunta,
e o FAISS encontra os fatos mais próximos no espaço vetorial.

In [ ]:
# ── Busca semântica ────────────────────────────────────────────────
def buscar(pergunta: str, k: int = 3) -> list[tuple[str, float]]:
    """Busca os k fatos mais similares à pergunta."""
    query_emb = gerar_embedding(pergunta).reshape(1, -1)
    distancias, indices = index.search(query_emb, k)

    resultados = []
    for i, (dist, idx) in enumerate(zip(distancias[0], indices[0])):
        # distância L2 → similaridade (0 = idêntico, maior = mais distante)
        sim = 1 - dist / 2  # conversão aproximada para [0, 1]
        resultados.append((fatos[idx], sim, idx))
    return resultados

# ── Demo 1: pergunta simples ──
pergunta = "Quem trabalhou com lógica e pensamento?"
print(f"❓ Pergunta: {pergunta}")
print("=" * 60)

for fato, sim, idx in buscar(pergunta, k=3):
    print(f"  [{sim:.3f}] #{idx:02d}: {fato}")
    print()

In [ ]:
# ── Demo 2: pergunta que conecta filosofia + computação ──
pergunta2 = "Como linguagem e significado se relacionam?"
print(f"❓ Pergunta: {pergunta2}")
print("=" * 60)

for fato, sim, idx in buscar(pergunta2, k=3):
    print(f"  [{sim:.3f}] #{idx:02d}: {fato}")
    print()

## 5. Agente RAG: LLM com ferramenta de busca

Agora a parte mais legal: criamos um **agente** com o `gemma4:e2b-it-qat`
e damos a ele uma **ferramenta** para buscar fatos no nosso banco.

O agente recebe uma pergunta, decide *se* precisa buscar, *o que* buscar,
e combina os fatos encontrados para responder.

⚠️ Nomes e descrições em **inglês** — modelos entendem tool calling melhor em inglês.

In [ ]:
# ── Agente RAG com ferramenta de busca ─────────────────────────────
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

@tool
def search_facts(query: str, k: int = 3) -> str:
    """Search a knowledge base of facts about philosophy and computer science.
    Returns the k most semantically similar facts to the query.
    Use this tool when you need to find information about philosophers,
    computer scientists, logic, consciousness, or the history of computing.
    You can call this tool multiple times with different queries to gather
    information from multiple angles before answering.

    Args:
        query: A natural language question or topic to search for.
        k: Number of facts to retrieve (default 3, max 5).
    """
    k = min(k, 5)
    resultados = buscar(query, k=k)
    if not resultados:
        return "No facts found."
    saida = []
    for i, (fato, sim, idx) in enumerate(resultados, 1):
        saida.append(f"Fact {i} (similarity {sim:.2f}): {fato}")
    return "\n\n".join(saida)

llm = ChatOpenAI(
    model="gemma4:e2b-it-qat",
    base_url="http://localhost:11434/v1",
    api_key="nao_precisa",
    temperature=0,
)

agente = create_agent(llm, [search_facts])

print("✅ Agente RAG criado!")
print("   LLM: gemma4:e2b-it-qat")
print("   Ferramenta: search_facts (busca semântica no banco de fatos)")

## 6. Testando o agente

Vamos fazer uma pergunta que **exige combinar mais de um fato**.
O agente precisa buscar, recuperar, e conectar informações.

In [ ]:
# ── Teste 1: pergunta que conecta dois fatos ──
pergunta_agente = "Quem imaginou que máquinas poderiam pensar antes de os computadores existirem?"

print(f"❓ Pergunta: {pergunta_agente}")
print("=" * 60)

resultado = agente.invoke({"messages": pergunta_agente})

# Mostrar o processo completo (Thought → Action → Observation)
for msg in resultado["messages"]:
    if msg.type == "human":
        print(f"\n👤 [{msg.type}] {msg.content}")
    elif msg.type == "ai" and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"🤖 [tool_call] {tc['name']}({tc['args']})")
    elif msg.type == "tool":
        print(f"🔧 [tool_result] {msg.content[:200]}...")
    elif msg.type == "ai" and msg.content:
        print(f"\n🤖 [resposta]\n{msg.content}")

print("\n" + "=" * 60)
print(f"📝 Resposta final: {resultado['messages'][-1].content}")

In [ ]:
# ── Teste 2: pergunta que requer múltiplas buscas ──
pergunta_agente2 = "Qual a relação entre lógica, álgebra e computação? Busque fatos sobre os três temas e conecte."

print(f"❓ Pergunta: {pergunta_agente2}")
print("=" * 60)

resultado2 = agente.invoke({"messages": pergunta_agente2})

for msg in resultado2["messages"]:
    if msg.type == "human":
        print(f"\n👤 [{msg.type}] {msg.content}")
    elif msg.type == "ai" and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"🤖 [tool_call] {tc['name']}({tc['args']})")
    elif msg.type == "tool":
        print(f"🔧 [tool_result] {msg.content[:200]}...")
    elif msg.type == "ai" and msg.content:
        print(f"\n🤖 [resposta]\n{msg.content}")

print("\n" + "=" * 60)
print(f"📝 Resposta final: {resultado2['messages'][-1].content}")

---

## 📝 Resumo

- **Embeddings**: texto → vetor de números que captura significado
- **Similaridade por cosseno**: ângulo entre vetores = quão parecidos são
- **Matryoshka**: embeddings truncáveis (768 → 256 dim economiza memória)
- **FAISS**: vector store rápido e simples (create → add → search)
- **Busca semântica**: pergunta → embedding → FAISS → fatos relevantes
- **Agente RAG**: LLM + ferramenta de busca = buscar antes de responder
- **Combinação**: o agente pode buscar múltiplas vezes e conectar fatos

**Próxima aula:** Apresentação final — hora de mostrar os projetos!